# 2. Diseño de Arquitectura y Entrenamiento

Este notebook documenta la evolución iterativa del modelo desde un baseline simple (V1) hasta el modelo final multi-input (V4), justificando cada decisión de diseño.

**Constraint**: ≤250,000 parámetros.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.utils import load_config, get_device
from src.utils.visualization import set_style
from src.data import create_dataloaders, NUM_TEXT_FEATURES
from src.models import build_model, count_parameters, WatchPriceCNN

set_style()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 2.1 Decisiones de arquitectura

### Componentes clave

**Depthwise Separable Convolutions**: Reducen drásticamente los parámetros respecto a convoluciones estándar. Para k=3, C_in=64, C_out=128: 4,672 vs 73,728 params (16× ahorro). Esto nos permite tener más filtros y bloques dentro del presupuesto de 250K.

**Squeeze-and-Excitation (SE) Blocks**: Atención a nivel de canal con coste negligible en parámetros (~2·C²/r). Permiten al modelo aprender qué filtros son más relevantes para cada imagen.

**Dual Conv per Block (V2+)**: Dos convoluciones consecutivas por bloque en lugar de una, duplicando el campo receptivo. Esto fue clave para que el Grad-CAM pasase de mirar puntos aislados a cubrir zonas amplias del reloj.

**Multi-Input (V3+)**: La imagen sola no puede distinguir marcas (requiere OCR). Añadimos brand embedding y features textuales del nombre como inputs auxiliares.

### Arquitectura final (V4)

```
Input imagen (3×224×224)
  → ConvBlock(3→32)   + SE + MaxPool    [standard conv para RGB]
  → ConvBlock(32→64)  + dual DWS + SE + MaxPool
  → ConvBlock(64→128) + dual DWS + SE + MaxPool
  → ConvBlock(128→256)+ dual DWS + SE + MaxPool
  → GlobalAvgPool → flatten (256-dim)

Input brand_idx → Embedding(70, 16) → (16-dim)

Input text_features → (21-dim binary vector)

Concat [256 + 16 + 21] = 293
  → FC(293→128) → GELU → FC(128→32) → GELU → FC(32→1)
```

In [ ]:
# Cargar config y construir modelo
config = load_config('../configs/base.yaml')
config['data']['metadata_path'] = '../data/metadata.csv'
config['data']['root_dir'] = '../data/raw'

_, _, _, brand2idx = create_dataloaders(config)
config['model']['num_brands'] = len(brand2idx)

model = build_model(config)

## 2.2 Evolución V1 → V4

### Resumen de cambios por versión

In [ ]:
evolution = pd.DataFrame({
    'Version': ['V1 (baseline)', 'V2', 'V3', 'V4 (final)'],
    'Params': ['107K', '200K', '201K', '226K'],
    'img_size': [192, 224, 224, 224],
    'base_filters': [32, 32, 32, 32],
    'Key Changes': [
        'Single conv, dropout=0.3, Huber loss, 128px→192px',
        'Dual conv, dropout=0.15, patience 15',
        '+ Brand embedding (70→16dim)',
        '+ 21 text features, CLAHE, MSE loss, LR=0.0005'
    ],
    'Motivation': [
        'Baseline con depthwise separable + SE',
        'Fix underfitting (val_loss < train_loss)',
        'Marca es el predictor #1 de precio',
        'Features textuales + mejor preprocessing'
    ],
    'R² (log)': [0.494, 0.579, 0.843, 0.872],
    'R² (dollar)': [0.339, 0.436, 0.751, 0.821],
    'MAE ($)': [181, 171, 94, 77],
    'MAPE (%)': [42.6, 40.2, 19.9, 18.3],
})
evolution

### Análisis de cada iteración

**V1 → V2 (underfitting fix)**: Las training curves de V1 mostraban val_loss por debajo de train_loss — señal clara de underfitting. Reducir dropout de 0.3 a 0.15 y añadir dual conv por bloque corrigió esto. El Grad-CAM pasó de activaciones puntuales a cubrir zonas amplias.

**V2 → V3 (brand embedding)**: El salto más grande (+71% en R² dollar). La marca es el factor dominante en el precio de un reloj, y la CNN no puede leer logos con 250K params. Un embedding de 16 dims con solo ~1,200 params extra resolvió esto.

**V3 → V4 (text features + refinamiento)**: 21 features binarias del nombre ("automatic", "chronograph", "gold", etc.) capturan señales de precio que ni la imagen ni la marca dan. CLAHE mejora el contraste de detalles. MSE en vez de Huber da gradientes más fuertes para relojes caros. LR más bajo permite convergencia más fina.

## 2.3 Training curves del modelo final (V4)

In [ ]:
# Cargar historial de entrenamiento
history_path = Path('../outputs/results/training_history.json')
if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].plot(history['train_loss'], label='Train', alpha=0.8)
    axes[0].plot(history['val_loss'], label='Validation', alpha=0.8)
    axes[0].set_title('Loss (MSE)')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    
    axes[1].plot(history['train_mae'], label='Train MAE', alpha=0.8)
    axes[1].plot(history['val_mae'], label='Val MAE', alpha=0.8)
    axes[1].set_title('MAE (log scale)')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    
    axes[2].plot(history['lr'], color='green')
    axes[2].set_title('Learning Rate (Cosine Annealing)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/v4_training_curves_detailed.png', dpi=150)
else:
    print('No se encontró historial. Ejecuta primero: python scripts/train.py')

## 2.4 Comparativa de métricas

In [ ]:
# Gráfico de evolución de R²
versions = ['V1', 'V2', 'V3', 'V4']
r2_log = [0.494, 0.579, 0.843, 0.872]
r2_dollar = [0.339, 0.436, 0.751, 0.821]
mae = [181, 171, 94, 77]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(versions, r2_log, color=['#85B7EB', '#85B7EB', '#5DCAA5', '#1D9E75'])
axes[0].axhline(0.85, color='red', linestyle='--', label='Objetivo (0.85)')
axes[0].set_title('R² (log scale)')
axes[0].set_ylim(0, 1)
axes[0].legend()
for i, v in enumerate(r2_log):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(versions, r2_dollar, color=['#85B7EB', '#85B7EB', '#5DCAA5', '#1D9E75'])
axes[1].set_title('R² (dollar scale)')
axes[1].set_ylim(0, 1)
for i, v in enumerate(r2_dollar):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

axes[2].bar(versions, mae, color=['#F09595', '#F09595', '#5DCAA5', '#1D9E75'])
axes[2].set_title('MAE ($)')
for i, v in enumerate(mae):
    axes[2].text(i, v + 3, f'${v}', ha='center', fontweight='bold')

plt.suptitle('Evolución de métricas V1 → V4', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/metrics_evolution.png', dpi=150)

## 2.5 Presupuesto de parámetros

In [ ]:
# Desglose de parámetros por componente
components = {
    'CNN Backbone': sum(p.numel() for p in model.features.parameters()),
    'Brand Embedding': sum(p.numel() for p in model.brand_embedding.parameters()),
    'FC Head': sum(p.numel() for p in model.head.parameters()),
    'GAP + otros': sum(p.numel() for p in model.pool.parameters()),
}
total = sum(components.values())

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(list(components.keys()), list(components.values()), color='steelblue')
ax.axvline(250000, color='red', linestyle='--', label='Límite (250K)')
ax.set_xlabel('Parámetros')
ax.set_title(f'Distribución de parámetros ({total:,} / 250,000)')
for bar, v in zip(bars, components.values()):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2, 
            f'{v:,} ({v/total*100:.0f}%)', va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/param_budget.png', dpi=150)

## 2.6 Técnicas de optimización y regularización

| Técnica | Configuración | Propósito |
|---------|---------------|----------|
| Early stopping | patience=25, min_delta=0.0005 | Prevenir sobreajuste |
| Cosine annealing | T₀=195, η_min=1e-6 | LR scheduling suave |
| Dropout | 0.15 (head), 0.075 (mid) | Regularización del FC head |
| Weight decay | 0.01 (AdamW) | L2 regularización implícita |
| Gradient clipping | max_norm=1.0 | Estabilidad de entrenamiento |
| CLAHE preprocessing | clip_limit=2.0 | Mejora contraste en todas las imágenes |
| Data augmentation | Flip, Rotate, Crop, Brightness, CoarseDropout | Generalización |